# 03 — Evaluation harness

This notebook defines **how anomaly detectors will be judged before any detector is built**. It:

- separates calibration, development and holdout truth physically;
- fixes one common score and alert interface;
- fixes alert-to-fault matching and operational metrics;
- proves the evaluator behaves correctly with adversarial controls.

It does not train a model. Notebook 04 will use this frozen interface.

## 1. Setup and sector switch

Change `SECTOR` when running the other canonical dataset. Everything else is common.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (
    Path("/content/drive/MyDrive/anomaly_detection")
    if IN_COLAB else Path.home() / "anomaly_detection_data"
)
DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or default_data_root
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB else Path.cwd(),
)).expanduser()

required_helpers = ["milestone1_core.py", "evaluation_core.py"]
missing_helpers = [name for name in required_helpers if not (NOTEBOOK_HOME / name).is_file()]
if missing_helpers:
    raise FileNotFoundError(
        f"Missing {missing_helpers} in {NOTEBOOK_HOME}. Copy them beside this notebook."
    )
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    CORE_VERSION, EVAL_VERSION, check_evaluation, fault_coverage,
    file_sha256, new_output_directory, read_json, write_json,
)
from evaluation_core import (
    ALERT_COLUMNS, PARTITIONS, SCORE_COLUMNS,
    evaluate_alerts, partition_truth, scores_to_alerts,
)

# Change this value when moving between sectors.
SECTOR = os.getenv("EVALUATION_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run1",
}
EVALUATION_RUN_IDS = {
    "telecom": "telecom_evaluation_v0_1_run1",
    "petrobras_3w": "petrobras_3w_evaluation_v0_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EVALUATION_RUN_ID = os.getenv("EVALUATION_RUN_ID", EVALUATION_RUN_IDS[SECTOR])
EVALUATION_VERSION = "0.1.0"
MIN_RELIABLE_FAULTS = 5
SAVE_OUTPUTS = os.getenv("SAVE_EVALUATION_OUTPUTS", "1") == "1"
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / EVALUATION_RUN_ID

display(pd.Series({
    "sector": SECTOR,
    "canonical_run": str(RUN_ROOT),
    "evaluation_output": str(OUTPUT_ROOT),
    "spec_core_version": CORE_VERSION,
    "spec_eval_version": EVAL_VERSION,
    "save_outputs": SAVE_OUTPUTS,
}, name="value").to_frame())

## 2. Read truth and choose the primary split

The notebook reads canonical truth—not native sector files. Telecom defaults to chronological partitions. 3W defaults to whole-well partitions. `EVALUATION_PRIMARY_SPLIT` can override the default only when that split table exists.

In [ ]:
if not RUN_ROOT.is_dir():
    raise FileNotFoundError(f"Run Notebook 01B first: {RUN_ROOT}")
if not (RUN_ROOT / "SPEC-EVAL").is_dir():
    raise FileNotFoundError("Notebook 03 requires a labelled canonical run")

run_manifest = read_json(RUN_ROOT / "run_manifest.json")
core_manifest = read_json(RUN_ROOT / "SPEC-CORE" / "manifest.json")
if core_manifest["contract_version"] != CORE_VERSION:
    raise ValueError("Canonical contract version does not match this notebook")

def read_optional(path, columns):
    return pd.read_parquet(path) if Path(path).is_file() else pd.DataFrame(columns=columns)

events = pd.read_parquet(RUN_ROOT / "SPEC-EVAL" / "fault_events.parquet")
intervals = pd.read_parquet(RUN_ROOT / "SPEC-EVAL" / "fault_entity_intervals.parquet")
conditions = read_optional(
    RUN_ROOT / "SPEC-EVAL" / "condition_states.parquet",
    ["entity_id", "start_ts", "end_ts", "condition_code", "label_source", "source_instance_id"],
)
registry = pd.read_parquet(RUN_ROOT / "SPEC-CORE" / "entity_registry.parquet")
time_partitions = read_optional(
    RUN_ROOT / "SPLITS" / "time_partitions.parquet",
    ["partition", "start_ts", "end_ts", "split_version"],
)
entity_partitions = read_optional(
    RUN_ROOT / "SPLITS" / "entity_partitions.parquet",
    ["entity_id", "partition", "split_version"],
)

for frame in (events, intervals, conditions, registry, time_partitions):
    time_columns = [name for name in frame.columns if name.endswith("_ts") or name in {"observed_from", "observed_to"}]
    for column in time_columns:
        frame[column] = pd.to_datetime(frame[column], utc=True, errors="coerce")

default_split = "time" if not time_partitions.empty else "entity"
PRIMARY_SPLIT = os.getenv("EVALUATION_PRIMARY_SPLIT", default_split)
if PRIMARY_SPLIT not in {"time", "entity"}:
    raise ValueError("EVALUATION_PRIMARY_SPLIT must be 'time' or 'entity'")
if PRIMARY_SPLIT == "time" and time_partitions.empty:
    raise FileNotFoundError("This run has no time partitions")
if PRIMARY_SPLIT == "entity" and entity_partitions.empty:
    raise FileNotFoundError("This run has no entity partitions")

evaluation_audit = check_evaluation(RUN_ROOT)
display(pd.Series({
    "primary_split": PRIMARY_SPLIT,
    "fault_events": len(events),
    "fault_entity_intervals": len(intervals),
    "condition_states": len(conditions),
    **evaluation_audit,
}, name="value").to_frame())

## 3. Partition truth physically

A fault is assigned using `observable_ts` (falling back to `onset_ts`). Cross-partition and non-observable faults remain in the audit but are excluded from strict scoring. `holdout_sealed` is not opened by Notebook 04.

In [ ]:
truth_by_partition, fault_audit, truth_summary = partition_truth(
    events, intervals, conditions, registry,
    primary_split=PRIMARY_SPLIT,
    time_partitions=time_partitions,
    entity_partitions=entity_partitions,
)
display(truth_summary)

exceptions = fault_audit.loc[~fault_audit["status"].eq("scoreable")]
if not exceptions.empty:
    print("Faults retained in the audit but excluded from strict scoring:")
    display(exceptions.sort_values(["status", "fault_type", "fault_id"]))

thin_holdout = truth_summary.loc[
    truth_summary["assigned_partition"].eq("holdout")
    & truth_summary["status"].eq("scoreable")
    & truth_summary["faults"].lt(MIN_RELIABLE_FAULTS)
]
if not thin_holdout.empty:
    print("WARNING — holdout fault-type estimates below five events are descriptive only.")
    display(thin_holdout)

secondary_entity_coverage = pd.DataFrame()
if PRIMARY_SPLIT == "time" and not entity_partitions.empty:
    secondary_entity_coverage = fault_coverage(RUN_ROOT, partition_table="entity_partitions")
    print("Secondary whole-entity fault coverage (not the primary Telecom split):")
    display(secondary_entity_coverage)

## 4. Common detector interface

Every future detector emits the same score table. Consecutive above-threshold scores become one alert. This separates **model scoring** from **operational alerting**.

In [ ]:
display(pd.DataFrame({
    "score_field": SCORE_COLUMNS,
    "meaning": ["time scored", "asset scored", "higher means more anomalous", "model/version name"],
}))
display(pd.DataFrame({
    "alert_field": ALERT_COLUMNS,
    "meaning": [
        "unique alert", "model/version name", "affected asset", "first qualifying score",
        "end of alert run", "time of largest score", "largest score", "scores in alert run",
    ],
}))

## 5. Frozen matching and reporting policy

- Match from the later of observable time and affected-entity interval start.
- Stop at the earlier of the fault end and affected-entity interval end.
- Use `alert_start` as detection time.
- One alert matches at most one fault; one fault gets one event-level credit.
- Later eligible alerts are duplicates.
- Report recall, pre-impact recall, precision, alert burden, latency, shared-fault recall and affected-entity coverage separately.
- Ratio metrics use Wilson 95% intervals. Fault types with fewer than five holdout events are descriptive only.

## 6. Adversarial controls

Tiny artificial scorers test the evaluator itself. A perfect scorer must be perfect; a late scorer must fail pre-impact recall; duplicates must increase burden; and a deliberately leaky scorer must fail when truth is absent.

In [ ]:
CONTROL_THRESHOLD = 0.80
CONTROL_CADENCE = pd.Timedelta(minutes=5)


def make_control_fixture():
    base = pd.Timestamp("2025-01-01", tz="UTC")
    events = pd.DataFrame([
        ("F-1", "fault_a", "asset-1", base + pd.Timedelta("15min"), base + pd.Timedelta("20min"), base + pd.Timedelta("30min"), base + pd.Timedelta("50min"), "cause-1"),
        ("F-2", "fault_b", "asset-1", base + pd.Timedelta("75min"), base + pd.Timedelta("80min"), base + pd.Timedelta("90min"), base + pd.Timedelta("110min"), "cause-2"),
        ("F-3", "shared_fault", "group-1", base + pd.Timedelta("135min"), base + pd.Timedelta("140min"), base + pd.Timedelta("150min"), base + pd.Timedelta("170min"), "cause-3"),
    ], columns=["fault_id", "fault_type", "domain_id", "onset_ts", "observable_ts", "impact_ts", "end_ts", "group_id"])
    intervals = pd.DataFrame([
        ("F-1", "asset-1", base + pd.Timedelta("15min"), base + pd.Timedelta("50min")),
        ("F-2", "asset-1", base + pd.Timedelta("75min"), base + pd.Timedelta("110min")),
        ("F-3", "asset-2", base + pd.Timedelta("135min"), base + pd.Timedelta("170min")),
        ("F-3", "asset-3", base + pd.Timedelta("135min"), base + pd.Timedelta("170min")),
    ], columns=["fault_id", "entity_id", "start_ts", "end_ts"])
    grid = pd.MultiIndex.from_product([
        ["asset-1", "asset-2", "asset-3"],
        pd.date_range(base, base + pd.Timedelta("3h"), freq=CONTROL_CADENCE),
    ], names=["entity_id", "event_ts"]).to_frame(index=False)
    grid["anomaly_score"] = 0.0
    grid["model_id"] = "control"
    entity_days = grid["entity_id"].nunique() * (
        grid["event_ts"].max() - grid["event_ts"].min() + CONTROL_CADENCE
    ).total_seconds() / 86400
    return events, intervals, grid, entity_days


def mark_alert(scores, entity_id, start):
    mask = scores["entity_id"].eq(entity_id) & scores["event_ts"].isin([start, start + CONTROL_CADENCE])
    scores.loc[mask, "anomaly_score"] = 1.0


def make_control_scores(kind, grid, events):
    scores = grid.copy()
    scores["model_id"] = kind
    if kind == "constant":
        return scores
    if kind == "random":
        scores["anomaly_score"] = np.random.default_rng(42).random(len(scores))
        return scores
    entity_for_fault = {"F-1": "asset-1", "F-2": "asset-1", "F-3": "asset-2"}
    for event in events.itertuples(index=False):
        start = event.observable_ts if kind in {"perfect", "duplicate"} else event.impact_ts + CONTROL_CADENCE
        mark_alert(scores, entity_for_fault[event.fault_id], start)
        if kind == "duplicate":
            mark_alert(scores, entity_for_fault[event.fault_id], event.impact_ts + CONTROL_CADENCE)
    return scores


def deliberately_leaky_scorer(truth_root, grid):
    events_path = Path(truth_root) / "fault_events.parquet"
    intervals_path = Path(truth_root) / "fault_entity_intervals.parquet"
    if not (events_path.is_file() and intervals_path.is_file()):
        raise FileNotFoundError("Truth is not mounted")
    truth_events = pd.read_parquet(events_path)
    first_entity = (
        pd.read_parquet(intervals_path).sort_values(["fault_id", "entity_id"])
        .drop_duplicates("fault_id").set_index("fault_id")["entity_id"]
    )
    scores = grid.copy()
    scores["model_id"] = "leaky"
    for event in truth_events.itertuples(index=False):
        mark_alert(scores, first_entity.loc[event.fault_id], event.observable_ts)
    return scores


control_events, control_intervals, control_grid, entity_days = make_control_fixture()
control_outputs = {}
for name in ("constant", "random", "perfect", "late", "duplicate"):
    scores = make_control_scores(name, control_grid, control_events)
    alerts = scores_to_alerts(scores, CONTROL_THRESHOLD, min_consecutive=2)
    control_outputs[name] = evaluate_alerts(
        alerts, control_events, control_intervals,
        exposure_entity_days=entity_days,
        min_reliable_faults=MIN_RELIABLE_FAULTS,
    )
    control_outputs[name]["alerts"] = alerts

with tempfile.TemporaryDirectory() as temporary:
    mounted = Path(temporary) / "truth"
    mounted.mkdir()
    control_events.to_parquet(mounted / "fault_events.parquet", index=False)
    control_intervals.to_parquet(mounted / "fault_entity_intervals.parquet", index=False)
    scores = deliberately_leaky_scorer(mounted, control_grid)
    alerts = scores_to_alerts(scores, CONTROL_THRESHOLD, min_consecutive=2)
    control_outputs["leaky"] = evaluate_alerts(
        alerts, control_events, control_intervals, exposure_entity_days=entity_days
    )
    control_outputs["leaky"]["alerts"] = alerts
    try:
        deliberately_leaky_scorer(Path(temporary) / "truth_not_mounted", control_grid)
    except FileNotFoundError:
        leakage_negative_control = "pass"
    else:
        raise AssertionError("Leaky scorer unexpectedly ran without truth")


def metric_value(control, metric):
    return control_outputs[control]["metrics"].set_index("metric").loc[metric, "value"]


assert metric_value("constant", "event_recall") == 0
assert metric_value("perfect", "event_recall") == 1
assert metric_value("perfect", "preimpact_event_recall") == 1
assert metric_value("perfect", "alert_precision") == 1
assert metric_value("late", "event_recall") == 1
assert metric_value("late", "preimpact_event_recall") == 0
assert metric_value("duplicate", "duplicate_alerts") > 0
assert leakage_negative_control == "pass"

control_summary = pd.concat([
    result["metrics"].assign(control=name) for name, result in control_outputs.items()
], ignore_index=True).pivot(index="control", columns="metric", values="value")
display(control_summary)
print("PASS — perfect alerts detect every fault before impact")
print("PASS — late alerts detect faults but fail pre-impact recall")
print("PASS — duplicate alerts increase burden without extra event credit")
print("PASS — constant and random controls use the same evaluator")
print("PASS — the leaky scorer fails when truth is not mounted")

## 7. Freeze policy and outputs

Development truth is available to Notebook 04 for threshold selection. Holdout truth stays sealed until Notebook 05.

In [ ]:
policy = {
    "evaluation_version": EVALUATION_VERSION,
    "spec_core_version": CORE_VERSION,
    "spec_eval_version": EVAL_VERSION,
    "sector": SECTOR,
    "primary_split": PRIMARY_SPLIT,
    "development_truth": "TRUTH/development",
    "holdout_truth": "TRUTH/holdout_sealed",
    "holdout_rule": "do_not_read_until_notebook_05",
    "score_schema": SCORE_COLUMNS,
    "alert_schema": ALERT_COLUMNS,
    "threshold_selection": "development_only_at_fixed_false_alert_budget",
    "alert_rule": {
        "persistence": "consecutive_scores",
        "detection_time": "alert_start",
        "new_alert_after_gap": "1.5_times_observed_score_cadence",
    },
    "matching_rule": {
        "start": "max(observable_ts, affected_entity_interval_start)",
        "end": "min(fault_end_ts, affected_entity_interval_end)",
        "one_alert_matches_at_most_one_fault": True,
        "one_fault_receives_at_most_one_event_credit": True,
        "later_eligible_alerts": "duplicate",
    },
    "latency_rule": "report_by_sector_because_observable_ts_provenance_differs",
    "false_alert_exposure": "canonical_observable_entity_days_in_partition",
    "condition_states": "preserved_but_not_primary_fault_metric_in_v0.1",
    "unscoreable_faults": "reported_and_excluded_from_primary_recall",
    "cross_partition_faults": "reported_and_excluded_from_strict_scoring",
    "rare_fault_type_rule": f"fewer_than_{MIN_RELIABLE_FAULTS}_events_is_descriptive_only",
    "primary_metrics": [
        "event_recall", "preimpact_event_recall", "alert_precision",
        "false_alerts_per_entity_day", "median_detection_delay_seconds",
        "p90_detection_delay_seconds", "shared_fault_recall", "entity_fault_coverage",
    ],
}
manifest = {
    "evaluation_version": EVALUATION_VERSION,
    "sector": SECTOR,
    "canonical_run": str(RUN_ROOT),
    "canonical_manifest_sha256": file_sha256(RUN_ROOT / "SPEC-CORE" / "manifest.json"),
    "canonical_fingerprint": core_manifest["fingerprint"],
    "primary_split": PRIMARY_SPLIT,
    "truth_rows": {
        partition: {name: len(frame) for name, frame in tables.items()}
        for partition, tables in truth_by_partition.items()
    },
    "fault_status_counts": fault_audit["status"].value_counts().sort_index().to_dict(),
    "control_scorers": sorted(control_outputs),
    "control_tests_passed": True,
    "leakage_negative_control": leakage_negative_control,
}


def write_truth_partition(root, partition, tables):
    name = "holdout_sealed" if partition == "holdout" else partition
    destination = root / "TRUTH" / name
    destination.mkdir(parents=True)
    for table_name, frame in tables.items():
        if not frame.empty:
            frame.to_parquet(destination / f"{table_name}.parquet", index=False)


if SAVE_OUTPUTS:
    with new_output_directory(OUTPUT_ROOT) as output:
        for partition, tables in truth_by_partition.items():
            write_truth_partition(output, partition, tables)
        fault_audit.to_parquet(output / "fault_partition_audit.parquet", index=False)
        truth_summary.to_parquet(output / "truth_summary.parquet", index=False)
        if not secondary_entity_coverage.empty:
            secondary_entity_coverage.to_parquet(output / "secondary_entity_coverage.parquet", index=False)
        controls_root = output / "CONTROLS"
        controls_root.mkdir()
        for name, result in control_outputs.items():
            result["alerts"].to_parquet(controls_root / f"{name}_alerts.parquet", index=False)
            result["alert_matches"].to_parquet(controls_root / f"{name}_matches.parquet", index=False)
            result["metrics"].to_parquet(controls_root / f"{name}_metrics.parquet", index=False)
        control_summary.reset_index().to_parquet(controls_root / "control_summary.parquet", index=False)
        write_json(output / "evaluation_policy.json", policy)
        write_json(output / "evaluation_manifest.json", manifest)
    print("Saved evaluation harness:", OUTPUT_ROOT)
else:
    print("SAVE_EVALUATION_OUTPUTS=False: results were not written")

## 8. Acceptance and handoff

A green notebook means the **evaluation rules** are ready. It says nothing yet about model quality.

In [ ]:
acceptance = pd.Series({
    "canonical_contract_verified": True,
    "primary_split": PRIMARY_SPLIT,
    "truth_physically_partitioned": True,
    "holdout_sealed_for_notebook_05": True,
    "one_alert_one_fault_policy": True,
    "grouped_fault_metrics": True,
    "ratio_uncertainty": "Wilson 95%",
    "control_scorers_passed": True,
    "leakage_negative_control": leakage_negative_control,
    "unscoreable_faults": int(fault_audit["status"].eq("unscoreable").sum()),
    "cross_partition_faults": int(fault_audit["status"].eq("cross_partition").sum()),
}, name="result")
display(acceptance.to_frame())
print("Development truth:", OUTPUT_ROOT / "TRUTH" / "development")
print("Sealed holdout:", OUTPUT_ROOT / "TRUTH" / "holdout_sealed")
print("Next: 04_GENERIC_BASELINE_MODELS.ipynb")